**RANSAC** (RANdom SAmple Consensus) implemented from scratch — no sklearn,
no scipy fitting routines. Only numpy for basic array math and matplotlib
for the live visualization.

The three-step loop maps directly onto the code:

    1) guess inliers        -> pick a random minimal sample of points
    2) compute model        -> fit a line through that sample
    3) score the model      -> count how many of ALL points agree with it
                                (i.e. lie within some distance threshold)

Repeat N times, keep the model with the best score.

Run this colab directly to see it animate.

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

**Generate synthetic data: a line y = m*x + b buried in noise + outliers**

In [14]:
def make_data(n_points=100, outlier_ratio=0.4, true_m=2.0, true_b=-5.0,
              x_range=(0, 50), noise_std=1.0, seed=42):
    rng = np.random.default_rng(seed)

    n_outliers = int(n_points * outlier_ratio)
    n_inliers = n_points - n_outliers

    # clean inliers: points near the true line
    xs_in = rng.uniform(*x_range, size=n_inliers)
    ys_in = true_m * xs_in + true_b + rng.normal(0, noise_std, size=n_inliers)

    # outliers: scattered anywhere in the bounding box, unrelated to the line
    y_range = (true_m * x_range[0] + true_b - 30, true_m * x_range[1] + true_b + 30)
    xs_out = rng.uniform(*x_range, size=n_outliers)
    ys_out = rng.uniform(*y_range, size=n_outliers)

    xs = np.concatenate([xs_in, xs_out])
    ys = np.concatenate([ys_in, ys_out])

    # shuffle so inliers/outliers are mixed (RANSAC shouldn't get to "cheat")
    order = rng.permutation(len(xs))
    return xs[order], ys[order]

**Core RANSAC pieces, written as plain functions (no library does this for us)**

In [15]:
def fit_line_two_points(p1, p2):
    """Step 2: compute model. A line through exactly 2 points, in
    implicit form a*x + b*y + c = 0 (handles vertical lines fine,
    unlike y = m*x + b)."""
    x1, y1 = p1
    x2, y2 = p2
    a = y2 - y1
    b = x1 - x2
    c = -(a * x1 + b * y1)
    norm = np.hypot(a, b)
    if norm == 0:
        return None  # degenerate: identical points, no line
    return a / norm, b / norm, c / norm  # normalized so distance formula is simple

In [16]:
def point_line_distance(xs, ys, model):
    """Perpendicular distance of every point to the line a*x+b*y+c=0."""
    a, b, c = model
    return np.abs(a * xs + b * ys + c)

In [17]:
def score_model(xs, ys, model, threshold):
    """Step 3: score the model by testing ALL data points against it."""
    distances = point_line_distance(xs, ys, model)
    inlier_mask = distances < threshold
    return inlier_mask, inlier_mask.sum()


def refit_on_inliers(xs, ys, inlier_mask):
    """Optional polish: once we know the inlier set, refit using a proper
    least-squares line over all of them (not just the original 2-point
    sample)."""
    x_in, y_in = xs[inlier_mask], ys[inlier_mask]
    x_mean, y_mean = x_in.mean(), y_in.mean()
    x_c, y_c = x_in - x_mean, y_in - y_mean

    # 2x2 scatter matrix, solved by hand (closed form eigen-decomposition
    # of a symmetric 2x2 matrix) instead of calling np.linalg.eig, so every
    # step of "compute model" stays visible/derivable.
    sxx = np.sum(x_c * x_c)
    syy = np.sum(y_c * y_c)
    sxy = np.sum(x_c * y_c)

    theta = 0.5 * np.arctan2(2 * sxy, sxx - syy)
    # direction vector of the best-fit line
    dx, dy = np.cos(theta), np.sin(theta)
    # normal vector (perpendicular to direction) gives us a, b directly
    a, b = -dy, dx
    c = -(a * x_mean + b * y_mean)
    norm = np.hypot(a, b)
    return a / norm, b / norm, c / norm

**How many iterations do we actually need?** Standard RANSAC result:

              log(1 - p)
     N  =  ------------------
           log(1 - (1-eps)^s)

 where p = desired probability that at least one of the N samples is# all-inliers, eps = outlier ratio, s = minimal sample size (2 for a line).
 (1-eps)^s is the chance a single random sample is clean; raising the
 complement of that to the N-th power and solving for N when we want
 that failure probability under 1-p gives this formula.

In [18]:
def required_iterations(outlier_ratio, sample_size=2, success_prob=0.99):
    eps = outlier_ratio
    p = success_prob
    prob_clean_sample = (1 - eps) ** sample_size
    if prob_clean_sample >= 1.0:
        return 1  # eps == 0, first sample is guaranteed clean
    numerator = np.log(1 - p)
    denominator = np.log(1 - prob_clean_sample)
    return int(np.ceil(numerator / denominator))

**RANSAC main loop, generating a log of every iteration for the animation**

In [19]:
def ransac_line(xs, ys, n_iterations=60, threshold=1.5, seed=0):
    rng = np.random.default_rng(seed)
    n = len(xs)

    best_model = None
    best_inlier_mask = np.zeros(n, dtype=bool)
    best_count = -1

    history = []  # one entry per iteration, for later animation

    for it in range(n_iterations):
        # --- Step 1: guess inliers (a minimal random sample) ---
        idx = rng.choice(n, size=2, replace=False)
        p1, p2 = (xs[idx[0]], ys[idx[0]]), (xs[idx[1]], ys[idx[1]])

        # --- Step 2: compute model from that guess ---
        model = fit_line_two_points(p1, p2)
        if model is None:
            continue

        # --- Step 3: score the model against ALL points ---
        inlier_mask, count = score_model(xs, ys, model, threshold)

        improved = count > best_count
        if improved:
            best_count = count
            best_model = model
            best_inlier_mask = inlier_mask

        history.append({
            "iteration": it,
            "sample_idx": idx,
            "model": model,
            "inlier_mask": inlier_mask,
            "count": count,
            "is_best_so_far": improved,
            "best_count_so_far": best_count,
        })

    # final polish using the winning inlier set
    refined_model = refit_on_inliers(xs, ys, best_inlier_mask)

    return {
        "best_model": best_model,
        "refined_model": refined_model,
        "best_inlier_mask": best_inlier_mask,
        "history": history,
    }

**Live visualization: replays the history frame by frame**

In [20]:
def line_endpoints(model, x_range):
    """Given a*x+b*y+c=0, return two endpoints spanning x_range for plotting.
    Handles near-vertical lines (b ~ 0) by solving for x given y instead."""
    a, b, c = model
    x0, x1 = x_range
    if abs(b) > 1e-8:
        y0 = -(a * x0 + c) / b
        y1 = -(a * x1 + c) / b
        return (x0, x1), (y0, y1)
    else:
        x_vert = -c / a
        return (x_vert, x_vert), (-1000, 1000)


def animate_ransac(xs, ys, result, x_range, interval_ms=180):
    history = result["history"]

    fig, (ax_main, ax_score) = plt.subplots(
        1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [2.2, 1]}
    )

    # --- main plot: points + current candidate line + best-so-far line ---
    ax_main.set_xlim(x_range[0] - 3, x_range[1] + 3)
    y_pad = 20
    ax_main.set_ylim(ys.min() - y_pad, ys.max() + y_pad)
    ax_main.set_title("RANSAC — live iterations")
    ax_main.set_xlabel("x")
    ax_main.set_ylabel("y")

    all_points_scatter = ax_main.scatter(xs, ys, c="lightgray", s=25, zorder=2,
                                          label="all points")
    sample_scatter = ax_main.scatter([], [], c="black", s=90, marker="x",
                                      zorder=5, label="random sample (step 1)")
    inlier_scatter = ax_main.scatter([], [], c="tab:orange", s=25, zorder=3,
                                      label="candidate's inliers")
    best_inlier_scatter = ax_main.scatter([], [], c="tab:green", s=30, zorder=3,
                                           label="best-so-far inliers")

    candidate_line, = ax_main.plot([], [], "r--", lw=1.5, zorder=4,
                                    label="candidate model (step 2)")
    best_line, = ax_main.plot([], [], "g-", lw=2.5, zorder=4,
                               label="best model so far")

    ax_main.legend(loc="upper left", fontsize=8)
    iter_text = ax_main.text(0.98, 0.02, "", transform=ax_main.transAxes,
                              ha="right", va="bottom", fontsize=10,
                              family="monospace")

    # --- side plot: inlier count per iteration (the "score" over time) ---
    ax_score.set_title("Score (inlier count) per iteration")
    ax_score.set_xlabel("iteration")
    ax_score.set_ylabel("# inliers")
    ax_score.set_xlim(0, len(history))
    max_count = max(h["count"] for h in history) if history else 1
    ax_score.set_ylim(0, max_count + 5)

    score_line, = ax_score.plot([], [], "o-", ms=3, color="tab:blue",
                                 label="this iteration's score")
    best_score_line, = ax_score.plot([], [], "-", lw=2, color="tab:green",
                                      label="best score so far")
    ax_score.legend(loc="lower right", fontsize=8)

    scores_x, scores_y, best_y = [], [], []

    def init():
        candidate_line.set_data([], [])
        best_line.set_data([], [])
        sample_scatter.set_offsets(np.empty((0, 2)))
        inlier_scatter.set_offsets(np.empty((0, 2)))
        best_inlier_scatter.set_offsets(np.empty((0, 2)))
        return (candidate_line, best_line, sample_scatter,
                inlier_scatter, best_inlier_scatter)

    def update(frame_idx):
        h = history[frame_idx]

        # step 1 visual: highlight the random sample
        sample_pts = np.column_stack([xs[h["sample_idx"]], ys[h["sample_idx"]]])
        sample_scatter.set_offsets(sample_pts)

        # step 2 visual: draw the candidate line through that sample
        (cx0, cx1), (cy0, cy1) = line_endpoints(h["model"], x_range)
        candidate_line.set_data([cx0, cx1], [cy0, cy1])

        # step 3 visual: show which points count as inliers for THIS model
        mask = h["inlier_mask"]
        inlier_scatter.set_offsets(np.column_stack([xs[mask], ys[mask]]))

        # running best model
        best_so_far_mask = None
        for hh in history[:frame_idx + 1]:
            if hh["is_best_so_far"]:
                best_so_far_mask = hh["inlier_mask"]
                best_model_so_far = hh["model"]
        if best_so_far_mask is not None:
            best_inlier_scatter.set_offsets(
                np.column_stack([xs[best_so_far_mask], ys[best_so_far_mask]])
            )
            (bx0, bx1), (by0, by1) = line_endpoints(best_model_so_far, x_range)
            best_line.set_data([bx0, bx1], [by0, by1])

        iter_text.set_text(
            f"iter {h['iteration']+1:3d}/{len(history)}\n"
            f"this model inliers: {h['count']:3d}\n"
            f"best so far:        {h['best_count_so_far']:3d}"
        )

        scores_x.append(h["iteration"])
        scores_y.append(h["count"])
        best_y.append(h["best_count_so_far"])
        score_line.set_data(scores_x, scores_y)
        best_score_line.set_data(scores_x, best_y)

        return (candidate_line, best_line, sample_scatter, inlier_scatter,
                best_inlier_scatter, iter_text, score_line, best_score_line)

    anim = FuncAnimation(fig, update, frames=len(history), init_func=init,
                          interval=interval_ms, blit=False, repeat=False)
    plt.tight_layout()

    # IMPORTANT: keep a hard reference on the figure itself, otherwise some
    # backends/environments garbage-collect the FuncAnimation before it gets
    # a chance to render a single frame (shows up as a silent "Animation was
    # deleted without rendering anything" warning).
    fig._ransac_anim_ref = anim
    return anim, fig

In [22]:
if __name__ == "__main__":
    import sys

    outlier_ratio = 0.4
    xs, ys = make_data(n_points=100, outlier_ratio=outlier_ratio)

    n_iters = required_iterations(outlier_ratio, sample_size=2, success_prob=0.99)
    print(f"required_iterations(eps={outlier_ratio}, s=2, p=0.99) = {n_iters}")

    result = ransac_line(xs, ys, n_iterations=n_iters, threshold=1.5, seed=1)

    print("Best model (a,b,c) from raw 2-point sample:", result["best_model"])
    print("Refined model (a,b,c) after least-squares refit on inliers:",
          result["refined_model"])
    print(f"Inliers found: {result['best_inlier_mask'].sum()} / {len(xs)}")

    anim, fig = animate_ransac(xs, ys, result, x_range=(xs.min(), xs.max()))

    # Some environments (certain IDEs, remote/headless setups, non-interactive
    # backends, Google Colab) never open a real window and plt.show() either
    # returns instantly or does nothing, so the animation would appear to do
    # nothing. Detect those cases and fall back to a renderable form instead
    # of silently producing no output.
    try:
        import google.colab  # noqa: F401
        in_colab = True
    except ImportError:
        in_colab = False

    backend = plt.get_backend().lower()
    non_interactive = "agg" in backend and "tkagg" not in backend

    if in_colab:
        # Colab has no display server at all — plt.show() is a no-op here.
        # Render the animation to embedded HTML/JS (play/pause/scrub controls)
        # and display it inline in the output cell instead.
        from IPython.display import HTML, display
        #print("Google Colab detected — rendering animation as embedded HTML "
        #      "(this takes a few seconds)...")
        plt.close(fig)  # prevent the static final frame from also printing
        display(HTML(anim.to_jshtml()))
    elif non_interactive:
        out_path = "ransac_animation.gif"
        print(f"Non-interactive backend ({plt.get_backend()}) detected — "
              f"saving animation to {out_path} instead of opening a window.")
        anim.save(out_path, writer="pillow", fps=8)
        print("Saved.")
    else:
        plt.show(block=True)
        sys.exit(0)

required_iterations(eps=0.4, s=2, p=0.99) = 11
Best model (a,b,c) from raw 2-point sample: (np.float64(0.8888417970085851), np.float64(-0.458214207429832), np.float64(-1.987505583961306))
Refined model (a,b,c) after least-squares refit on inliers: (np.float64(-0.8949015526695274), np.float64(0.44626361159035705), np.float64(2.418129400890063))
Inliers found: 63 / 100
